## 2.3 构建三层 ConvLSTM 网络

把 2.2 的单层 `ConvLSTM` 改成三层堆叠：当前层的 $h_t$ 当作后一层的 $x_t$。

输入仍是 `(seq_len, batch, channels, height, width)`。

模型参数：

- `input_dim`：输入特征维数
- `kernel_size` / `filter_size`：卷积核尺寸
- `hidden_dim`：隐藏层状态的维数
- `num_layers`：层，理解为网络深度（这里固定为 3）

In [ ]:
import torch
import torch.nn as nn


class CLSTM_cell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size):
        super(CLSTM_cell, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size)
        self.kernel_size = kernel_size
        self.padding = (kernel_size[0] // 2, kernel_size[1] // 2)
        self.conv = nn.Conv2d(
            in_channels=input_dim + hidden_dim,
            out_channels=4 * hidden_dim,
            kernel_size=kernel_size,
            padding=self.padding,
        )

    def forward(self, input_tensor, hidden_state):
        h_cur, c_cur = hidden_state
        combined = torch.cat([input_tensor, h_cur], dim=1)
        combined_conv = self.conv(combined)
        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)
        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)
        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, (h_next, c_next)

In [ ]:
class ConvLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, num_layers=3):
        super(ConvLSTM, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # 三层：第 0 层读 x_t，第 1、2 层读上一层的 h_t
        cell_list = []
        for i in range(self.num_layers):
            cur_input_dim = input_dim if i == 0 else hidden_dim
            cell_list.append(CLSTM_cell(cur_input_dim, hidden_dim, kernel_size))
        self.cell_list = nn.ModuleList(cell_list)

    def _init_hidden(self, batch_size, image_size):
        height, width = image_size
        hidden_state = []
        for i in range(self.num_layers):
            # device=self.conv.weight.device: 选择设备
            device = self.cell_list[i].conv.weight.device
            # 初始化返回的隐状态与后面使用的隐状态组成成分一样；
            # 当前层的 h_t 会当作后一层的 x_t，所以每一层各自一份 (h, c)
            hidden_state.append((
                torch.zeros(batch_size, self.hidden_dim, height, width, device=device),
                torch.zeros(batch_size, self.hidden_dim, height, width, device=device),
            ))
        return hidden_state

    def forward(self, inputs=None, hidden_state=None):
        # 取出图片的数据，供下面初始化使用
        s, b, _, h, w = inputs.size()
        if hidden_state is None:
            # 初始化 hidden_state，利用后面和 lstm 单元中的初始化函数
            hidden_state = self._init_hidden(batch_size=b, image_size=(h, w))

        cur_layer_input = inputs
        last_state_list = []
        for layer_idx in range(self.num_layers):
            output_inner = []
            layer_state = hidden_state[layer_idx]
            for t in range(s):
                # 每一个时间步都更新 hidden_state 中的 h, c
                # 当前时间步的输出 hidden_state，又作为下一时间步的输入 hidden_state
                # 当前层的 h_t 会当作后一层的 x_t 作为输入使用
                hy, layer_state = self.cell_list[layer_idx](cur_layer_input[t, ...], layer_state)
                output_inner.append(hy)
            cur_layer_input = torch.stack(output_inner)
            last_state_list.append(layer_state)

        return cur_layer_input, last_state_list

In [ ]:
# 用法与单层版本相同，默认已经是三层
seq_len, batch, channels, height, width = 10, 2, 1, 32, 32
model = ConvLSTM(input_dim=channels, hidden_dim=16, kernel_size=3)
x = torch.randn(seq_len, batch, channels, height, width)
y, states = model(x)
print(y.shape)          # (10, 2, 16, 32, 32)
print(len(states))      # 3
print(states[-1][0].shape)  # (2, 16, 32, 32)